In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from optbinning import BinningProcess
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task2')
RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task2/data/IT.csv')
oot = pl.read_csv('task2/data/OOT.csv')

labeled = it.filter(pl.col('TARGET').is_not_null())
print(f'Labeled: {labeled.shape[0]}, Target rate: {labeled["TARGET"].mean():.4f}')

var_cols = [c for c in it.columns if c.startswith('VARIABLE_')]
cat_cols = [c for c in var_cols if it[c].n_unique() <= 100]
num_cols = [c for c in var_cols if c not in cat_cols]
print(f'Numerical: {len(num_cols)}, Categorical: {len(cat_cols)}')

In [ ]:
cutoff = labeled.sort('TIME')['TIME'].quantile(0.8)
train = labeled.filter(pl.col('TIME') <= cutoff)
val = labeled.filter(pl.col('TIME') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')
print(f'Train target: {train["TARGET"].mean():.4f}, Val target: {val["TARGET"].mean():.4f}')

In [ ]:
feat_cols = var_cols
X_train = train.select(feat_cols).to_pandas()
y_train = train['TARGET'].to_numpy().astype(int)
X_val = val.select(feat_cols).to_pandas()
y_val = val['TARGET'].to_numpy().astype(int)
X_oot = oot.select(feat_cols).to_pandas()

for c in cat_cols:
    for df in [X_train, X_val, X_oot]:
        df[c] = df[c].astype(str).fillna('__missing__')

X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()
X_oot_lgb = X_oot.copy()

enc_map = {}
for c in cat_cols:
    le = LabelEncoder()
    all_vals = list(set(
        X_train_lgb[c].unique().tolist() + X_val_lgb[c].unique().tolist() + X_oot_lgb[c].unique().tolist()
    ))
    le.fit(all_vals)
    for df in [X_train_lgb, X_val_lgb, X_oot_lgb]:
        df[c] = le.transform(df[c])
    enc_map[c] = le

cat_indices = [feat_cols.index(c) for c in cat_cols]
print(f'Cat indices: {cat_indices}')

In [ ]:
bp = BinningProcess(
    variable_names=feat_cols,
    categorical_variables=cat_cols,
    min_prebin_size=0.02,
    min_bin_size=0.05,
    max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp.fit(X_train_lgb.values, y_train)
selected = list(bp.get_support(names=True))
print(f'Selected (IV>=0.02): {len(selected)}')

summary = bp.summary().sort_values('iv', ascending=False)
print(summary[['name', 'iv', 'dtype']].head(20).to_string())

In [ ]:
X_tr_woe = bp.transform(X_train_lgb.values, metric='woe')
X_va_woe = bp.transform(X_val_lgb.values, metric='woe')
X_oo_woe = bp.transform(X_oot_lgb.values, metric='woe')

best_gini_lr = 0
best_lr = None
best_C = None

for C in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]:
    lr = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
    lr.fit(X_tr_woe, y_train)
    p = lr.predict_proba(X_va_woe)[:, 1]
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'WOE+LR C={C}: val_gini={g:.4f}')
    if g > best_gini_lr:
        best_gini_lr, best_lr, best_C = g, lr, C

p_lr_val = best_lr.predict_proba(X_va_woe)[:, 1]
val_gini_lr = best_gini_lr
print(f'\nBest LR: C={best_C}, val_gini={val_gini_lr:.4f}')

In [ ]:
configs = [
    {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 100, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 0.0, 'reg_lambda': 0.0},
    {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 200, 'feature_fraction': 0.8, 'bagging_fraction': 0.9, 'bagging_freq': 3, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 15, 'learning_rate': 0.01, 'min_child_samples': 300, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 7, 'learning_rate': 0.05, 'min_child_samples': 500, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 2.0, 'reg_lambda': 2.0},
    {'num_leaves': 31, 'learning_rate': 0.01, 'min_child_samples': 200, 'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 0.5, 'reg_lambda': 0.5},
    {'num_leaves': 7, 'learning_rate': 0.01, 'min_child_samples': 300, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
]

best_lgb_gini = 0
model = None

for i, cfg in enumerate(configs):
    params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'feature_pre_filter': False, **cfg}
    lgb_tr = lgb.Dataset(X_train_lgb, y_train, free_raw_data=False, categorical_feature=cat_indices)
    lgb_va = lgb.Dataset(X_val_lgb, y_val, reference=lgb_tr, free_raw_data=False)
    m = lgb.train(params, lgb_tr, num_boost_round=3000, valid_sets=[lgb_va],
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    p = m.predict(X_val_lgb)
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'Config {i}: val_gini={g:.4f} (leaves={cfg["num_leaves"]}, lr={cfg["learning_rate"]}, iters={m.best_iteration})')
    if g > best_lgb_gini:
        best_lgb_gini, model = g, m

p_lgb_val = model.predict(X_val_lgb)
val_gini_lgb = best_lgb_gini
print(f'\nBest LGB: val_gini={val_gini_lgb:.4f}')

In [ ]:
best_ens_g = 0
best_w = 0

for w in np.arange(0.0, 1.05, 0.05):
    p_ens = w * p_lr_val + (1 - w) * p_lgb_val
    g = 2 * roc_auc_score(y_val, p_ens) - 1
    if g > best_ens_g:
        best_ens_g, best_w = g, w

print(f'Best ensemble: w_lr={best_w:.2f}, val_gini={best_ens_g:.4f}')

results = {'WOE_LR': val_gini_lr, 'LGB_raw': val_gini_lgb, 'Ensemble': best_ens_g}
best_name = max(results, key=results.get)
val_gini = results[best_name]
print(f'Best overall: {best_name} = {val_gini:.4f}')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v:.4f}')

In [ ]:
if best_name == 'WOE_LR':
    p_oot_final = best_lr.predict_proba(X_oo_woe)[:, 1]
elif best_name == 'LGB_raw':
    p_oot_final = model.predict(X_oot_lgb)
else:
    p_oot_final = best_w * best_lr.predict_proba(X_oo_woe)[:, 1] + (1 - best_w) * model.predict(X_oot_lgb)

preds = pl.DataFrame({'ID_APPLICATION': oot['ID_APPLICATION'], 'SCORE': p_oot_final})
preds.write_csv('task2/predictions.csv')
print(f'OOT predictions saved: {preds.shape}')

with mlflow.start_run(run_name='v4_label_enc_cats'):
    mlflow.log_param('model', best_name)
    mlflow.log_param('n_features', len(feat_cols))
    mlflow.log_param('n_selected_woe', len(selected))
    mlflow.log_metric('val_gini', val_gini)
    mlflow.log_metric('val_gini_lr', val_gini_lr)
    mlflow.log_metric('val_gini_lgb', val_gini_lgb)
    mlflow.log_metric('val_gini_ens', best_ens_g)
    mlflow.set_tag('task', 'task2')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')